In [1]:
import pandas as pd
import re
import time
import datetime

In [2]:
import requests
from bs4 import BeautifulSoup
from urllib.request import urlopen, Request
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

# 環境設定

In [3]:
KAI_URL = "https://www.hoshinoresorts.com/sp/kaitabi20s/"
KAI_TSUGARU_URL = "https://hoshinoresorts.com/JA/hotels/0000000114/plans/0000000431?checkIn=2023%2F12%2F04&stay=1&a=2&b=0&c=0&d=0"

# 関数定義

# 分析

## driverの設定

In [4]:
option = Options()                         
option.add_argument('--headless')  
driver = webdriver.Chrome(options=option)

## 各施設へのリンクを取得

In [5]:
driver.get(KAI_URL)
# reserveボタンをクリック
elem = driver.find_element(By.XPATH, '//*[@id="js-reservebtn"]')
elem.click()

In [6]:
html = driver.page_source.encode('utf-8')
soup = BeautifulSoup(html, "html.parser")
results = soup.find_all(class_= "cm-reserve__bodyListItem js-reserve__item")

hotel_names = []
links = []
for result in results:
    hotel_name = result.find("h4").text
    link = result.find("a").get("href")
    print(hotel_name)
    print(f'リンク：{link}')
    hotel_names.append(hotel_name)
    links.append(link)

界 ポロト
リンク：https://hoshinoresorts.com/plans/JA/0000000129/0000000040
界 津軽
リンク：https://hoshinoresorts.com/plans/JA/0000000114/0000000431
界 川治
リンク：https://hoshinoresorts.com/plans/JA/0000000119/0000000236
界 鬼怒川
リンク：https://hoshinoresorts.com/plans/JA/0000000121/0000000173
界 日光
リンク：https://hoshinoresorts.com/plans/JA/0000000118/0000000253
界 箱根
リンク：https://hoshinoresorts.com/plans/JA/0000000117/0000000338
界 仙石原
リンク：https://hoshinoresorts.com/plans/JA/0000000124/0000000025
界 アンジン
リンク：https://hoshinoresorts.com/plans/JA/0000000122/0000000092
界 伊東
リンク：https://hoshinoresorts.com/plans/JA/0000000108/0000000523
界 遠州
リンク：https://hoshinoresorts.com/plans/JA/0000000113/0000000452
界 アルプス
リンク：https://hoshinoresorts.com/plans/JA/0000000123/0000000079
界 松本
リンク：https://hoshinoresorts.com/plans/JA/0000000104/0000000491
界 加賀
リンク：https://hoshinoresorts.com/plans/JA/0000000103/0000000506
界 玉造
リンク：https://hoshinoresorts.com/plans/JA/0000000106/0000000512
界 出雲
リンク：https://hoshinoresorts.com/plans/JA/0000000132

## 空室情報を取得

In [7]:
# 予約可能日を確認
last_reservation_date = (datetime.datetime.now() + datetime.timedelta(days=44)).date()

reservable_info = {}
for hotel_name, link in zip(hotel_names, links):
    driver.get(link)
    time.sleep(5)
    html = driver.page_source.encode('utf-8')
    soup = BeautifulSoup(html, "html.parser")
    calendars = soup.find_all(class_= "c-calendar") # 2ヶ月分のカレンダーを取得

    # カレンダーから空いている日，部屋数を取得 ({ホテル名： {空室年月日：　空室数}})
    dates = []
    room_cnts = []
    for calendar in calendars: 
        ym = calendar.find_all(class_="header")[0].text
        reservable_days = calendar.find_all(class_ = "date date-defalut")
        reservable_room_cnts = calendar.find_all(class_ = "roomcount")
        if reservable_days and reservable_room_cnts:
            dates += [ym + day.text.replace("\n", "").replace(" ", "") + "日" for day in reservable_days]
            room_cnts += [room_cnt.text for room_cnt in reservable_room_cnts]
    reservable_info[hotel_name] = {d: r for d, r in zip(dates, room_cnts)}

In [8]:
print(f"{last_reservation_date.strftime('%Y月%m月%d日')}まで予約可能\n")
text = ""
for k, v in reservable_info.items():
    if v:
        text += f"{k}\n"
        for reservable_date, room_cnt in v.items():
            text += f"{reservable_date}: {room_cnt} \n"
print(text)

2024月02月05日まで予約可能

界 川治
2024年2月5日: 1 
界 アルプス
2024年1月1日: 3 
2024年1月3日: 2 
界 出雲
2024年1月10日: 1 
2024年1月11日: 1 
2024年1月16日: 1 
界 長門
2024年1月23日: 1 
2024年1月31日: 1 
界 雲仙
2024年1月23日: 2 
2024年1月24日: 1 
2024年1月30日: 1 
2024年1月31日: 2 
2024年2月1日: 1 

